# Nearest Neighbour Classification

In [1]:
import scipy.io
import numpy as np
import sklearn
from sklearn.feature_extraction import image
import matplotlib.pyplot as plt
from helper_functions import *

## Dataset - Data

In [2]:
data = scipy.io.loadmat('Data/data.mat')
faces = np.array(data['face'])
#number of classes 
M = 2
N = 400 # 3 per class (ignoring 3rd image in each class, done in next cell)

   ### Split Data

In [3]:

#select images 2 and 3 from each subject for training
faces_flat = faces.reshape(-1,faces.shape[2])
#ignore image 3 (illumination) of all classes (creates a dataset of 400 samples)
idx = []
for n in range(600):
    if n%3 == 0 or n%3 == 1:
        idx = idx + [n]
faces_flat = faces_flat[:,idx]
train_idx =[]
test_idx = []
print(faces_flat.shape)
#create training set (75% = 300)
training_faces = faces_flat[:,:300]
test_faces = faces_flat[:,300:400]

# #rn test and training set have neutral face and expression alternating
#Create label Arrays
training_labels = [-1,1]*150
test_labels = [-1,1]*50

# training_faces = np.vstack(([-1,1]*150, training_faces))
# test_faces = np.vstack(([-1,1]*50, test_faces))

#datasets have alternating neutral and expression pictures
#rearrange saw that all the neutral come first then expression
training_faces = np.hstack((training_faces[:,::2], training_faces[:,1::2])) #0 -149 neutral, 150 - 299 expression
test_faces = np.hstack((test_faces[:,::2], test_faces[:,1::2])) #0-49 neutral, 50 - 99 expression
print(training_faces.shape, test_faces.shape)

(504, 400)
(504, 300) (504, 100)


### Perform PCA

In [7]:
#reduce dimensions from 504 to 1 (only 2 classes, so 1 dimension is the only possibility).
A, train_mean = PCA(training_faces, 50)
X_PCA = A @ (training_faces - train_mean)
X_test_PCA = A @ (test_faces - train_mean)
print(X_PCA.shape)


(50, 300)


### Perform MDA

In [8]:
#We have two classes here
A,class_means,sigma_w = MDA(X_PCA, 1, M)
X_MDA = A @ X_PCA
X_test_MDA = A @ X_test_PCA

### Bayesian Classifier

In [10]:
X_train = X_MDA
X_test = X_test_MDA
M = 2
#Bayes Classifier
#Training error
classified = Gaussian_Bayes(X_train, X_train,M)
#create label array for finding error
y_train = np.concatenate((np.zeros((1,150)), np.ones((1,150))), axis = 1)
#find error
train_error = 1/y_train.shape[1]*sum(a!=b for (a,b) in zip(classified.flatten(),y_train.flatten()))
print(train_error)

#Test error
classified = Gaussian_Bayes(X_train, X_test,M)
#create label array for finding error
y_test = np.concatenate((np.zeros((1,50)), np.ones((1,50))), axis = 1)
#find error
test_error = 1/y_test.shape[1]*sum(a!=b for (a,b) in zip(classified.flatten(),y_test.flatten()))
print(test_error)

0.06333333333333334
0.11
